# Task 3 trên Kaggle GPU T4 ×2 — không cần API key

Bản v3-fixed: giảm max_input_tokens (4096), max_new_tokens (384), bỏ stress test VRAM thừa, thêm gc.collect + empty_cache giữa smoke và main run. Không cần file recover.

Chọn Settings → Accelerator → GPU T4 ×2, bật Internet. Add Input bộ mã nguồn task3_kaggle_source (ZIP hoặc đã giải nén), sample_dataset đủ 32 mẫu, public test và requested levels.

Model: Qwen/Qwen3-4B-Instruct-2507, NF4 4-bit, FP16, tắt thinking. Mỗi GPU chạy một bản model trên một nửa dữ liệu.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, os, shutil
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
REPO = WORK / 'LLM-APA-task3'

# Kaggle có thể tự giải nén ZIP khi upload.
sources = sorted({p.parent.parent for p in INPUT.rglob('scripts/predict.py')
                  if (p.parent.parent / 'src/llm_grading').is_dir()
                  and (p.parent.parent / 'tasks/task3_feedback/prompts/v004.txt').is_file()})
archives = list(INPUT.rglob('task3_kaggle_source.zip'))
if len(sources) == 1:
    shutil.copytree(sources[0], REPO, dirs_exist_ok=True)
    print('Đã copy mã nguồn từ:', sources[0])
elif not sources and len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(REPO)
    print('Đã giải nén:', archives[0])
else:
    raise ValueError(f'Cần một bộ mã nguồn duy nhất. Thư mục: {sources}; ZIP: {archives}')

assert (REPO/'tasks/task3_feedback/prompts/v006.txt').is_file(), 'Cập nhật task3_kaggle_source.zip mới (prompt v006)'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyyaml', 'scikit-learn', 'tqdm', 'transformers>=4.56,<5', 'accelerate>=1.0,<2', 'bitsandbytes>=0.46,<1'], check=True)
sys.path[:0] = [str(REPO), str(REPO / 'src')]

def run(script, *args):
    subprocess.run([sys.executable, str(REPO/'scripts'/script), *map(str,args)], check=True)


Nếu cell cài thư viện yêu cầu restart kernel, hãy restart rồi chạy lại các cell trước khi tiếp tục. Giữ PyTorch CUDA có sẵn của Kaggle, không chạy uv sync --all-groups ở notebook này.


In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Chưa bật GPU'
assert torch.cuda.device_count() == 2, 'Chọn GPU T4 x2 trong Settings'
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), 'GiB')


In [ ]:
# Kiểm tra VRAM nhẹ — không cần stress test 8192-token attention.
import gc, torch
for gpu_id in range(2):
    p = torch.cuda.get_device_properties(gpu_id)
    free = p.total_memory - torch.cuda.memory_allocated(gpu_id)
    print(f'GPU {gpu_id}: {p.name} — {free / 2**30:.1f} GiB free / {p.total_memory / 2**30:.1f} GiB total')
    assert free > 8 * 2**30, f'GPU {gpu_id}: cần ít nhất 8 GiB free'
print('VRAM check OK')


In [ ]:
# ── Đường dẫn & Dữ liệu ────────────────────────────────────────────────────
OUT = WORK / 'task3-qwen3-4b-t4-v3'
OUT.mkdir(parents=True, exist_ok=True)

# Giải nén public_test.zip nếu được upload dưới dạng ZIP
public_archives = list(INPUT.rglob('public_test.zip'))
if public_archives:
    with zipfile.ZipFile(public_archives[0]) as z:
        z.extractall(WORK / 'public-data')

# Tự động tìm file task3_feedback.json (phân biệt giữa train và public test)
files = list(INPUT.rglob('task3_feedback.json')) + list((WORK / 'public-data').rglob('task3_feedback.json'))

def choose_feedback(is_training):
    candidates = []
    for p in files:
        if '__pycache__' in str(p):
            continue
        try:
            rows = json.loads(p.read_text(encoding='utf-8-sig'))['samples']
            if bool(rows and 'output' in rows[0]) == is_training:
                candidates.append(p)
        except Exception:
            pass
    assert candidates, f"Không tìm thấy file task3_feedback.json (is_training={is_training})"
    return candidates[0]

PUBLIC_JSON = choose_feedback(False)
print("Public test:", PUBLIC_JSON)

# Requested levels JSON
_level_jsons = sorted(INPUT.rglob("task3_requested_levels*.json"))
assert _level_jsons, "Chưa add task3_requested_levels JSON vào Input"
_raw_levels = json.loads(_level_jsons[0].read_text(encoding="utf-8-sig"))
if isinstance(_raw_levels, list):
    levels = {r['sample_id']: r['requested_level'] for r in _raw_levels}
elif isinstance(_raw_levels, dict):
    levels = _raw_levels
else:
    raise ValueError(f"Định dạng levels không hỗ trợ: {type(_raw_levels)}")
print("Levels:", len(levels), "bài —", dict(list(levels.items())[:3]), "...")

# Kiểm tra khớp sample_id giữa public test và requested levels
public_samples = json.loads(PUBLIC_JSON.read_text(encoding="utf-8-sig"))["samples"]
assert set(levels) == {r['sample_id'] for r in public_samples}, "Sample IDs không khớp giữa public test và levels"

# ── Config ─────────────────────────────────────────────────────────────────
cfg = {
    "task": "task3",
    "model": {
        "name": "Qwen/Qwen3-4B-Instruct-2507",
        "cache_dir": str(WORK / "hf_cache"),
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "cache_path": str(OUT / "generation_cache.jsonl"),
    },
    "generation": {
        "max_input_tokens": 4096,       # T4-safe (8192 → OOM)
        "max_new_tokens": 384,           # feedback chỉ cần ~200-300 tokens
        "attention_query_chunk_size": 128,
        "do_sample": False,
        "temperature": 0,
        "num_beams": 1,
        "enable_thinking": False,        # tắt CoT để tiết kiệm tokens
    },
    "prompt": {
        "task3": {
            "version": "v006",
            "prompt_dir": str(REPO / "tasks/task3_feedback/prompts"),
        }
    },
    "retrieval": {
        "enabled": False,
    },
    "saving": {
        "output_dir": str(OUT),
        "responses_file": "responses.jsonl",
        "log_file": "predict.log",
    },
    "pipeline": {
        "max_retries": 2,
        "fallback_on_error": True,
    },
}

CONFIG = OUT / "config.json"
CONFIG.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("Config saved →", CONFIG)
print(f"max_input_tokens={cfg['generation']['max_input_tokens']}, "
      f"max_new_tokens={cfg['generation']['max_new_tokens']}, "
      f"chunk_size={cfg['generation']['attention_query_chunk_size']}")


In [ ]:
# Smoke trên GPU 0. Không nạp model vào kernel notebook.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false',
           PYTHONUNBUFFERED='1')
subprocess.run([sys.executable, '-u', str(REPO/'scripts/predict.py'),
    '--config', str(CONFIG), '--input', str(PUBLIC_JSON), '--max-samples','3',
    '--output',str(OUT/'smoke_predictions.json')], env=env, check=True)
print((OUT/'smoke_predictions.json').read_text(encoding='utf-8'))

# Giải phóng VRAM sau smoke test
import gc
gc.collect()
torch.cuda.empty_cache()
for gpu_id in range(2):
    free = torch.cuda.get_device_properties(gpu_id).total_memory - torch.cuda.memory_allocated(gpu_id)
    print(f'GPU {gpu_id} sau smoke: {free / 2**30:.1f} GiB free')


## Chạy cả hai GPU

Hai worker chạy shard 1/2 và 2/2. Mỗi shard có config, log và cache riêng. Cell hiển thị các dòng tiến độ từ cả hai GPU. Giữ lại toàn bộ thư mục task3-qwen3-4b-t4-v3 nếu cần chạy lại: cache chỉ tái sử dụng request đã hoàn tất trong cùng cấu hình/model revision.


In [ ]:
import copy
import selectors

workers = []
selector = selectors.DefaultSelector()
try:
    for gpu in range(2):
        shard_dir = OUT/f'shard-{gpu+1}'
        shard_dir.mkdir(exist_ok=True)
        shard_cfg = copy.deepcopy(cfg)
        shard_cfg['saving']['output_dir'] = str(shard_dir)
        shard_cfg['model']['cache_path'] = str(shard_dir/'generation_cache.jsonl')
        config_path = shard_dir/'config.json'
        config_path.write_text(json.dumps(shard_cfg,ensure_ascii=False,indent=2),encoding='utf-8')
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1')
        cmd = [sys.executable,'-u',str(REPO/'scripts/predict.py'),
            '--config',str(config_path),'--input',str(PUBLIC_JSON),
            '--shard',f'{gpu+1}/2','--resume','--output',str(shard_dir/'predictions.json')]
        p = subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        workers.append(p)
        selector.register(p.stdout, selectors.EVENT_READ, gpu)
    while selector.get_map():
        for key, _ in selector.select(timeout=1):
            line = key.fileobj.readline()
            if line:
                print(f'[GPU {key.data}] {line}',end='',flush=True)
            else:
                selector.unregister(key.fileobj)
                key.fileobj.close()
    codes = [p.wait() for p in workers]
    assert codes == [0,0], f'Worker lỗi: {codes}; xem log phía trên, chưa đóng gói kết quả.'
finally:
    selector.close()
    for p in workers:
        if p.poll() is None:
            p.terminate()
    for p in workers:
        p.wait()


In [ ]:
run('merge_predictions.py','--task','task3',
    '--shards',OUT/'shard-1/predictions.json',OUT/'shard-2/predictions.json',
    '--input',PUBLIC_JSON,'--output',OUT/'predictions.json')
pred=json.loads((OUT/'predictions.json').read_text(encoding='utf-8'))
assert {r['sample_id'] for r in pred} == set(levels)
assert all(r['output']['feedback_level']==levels[r['sample_id']] for r in pred)
run('validate_predictions.py','--task','task3','--input',OUT/'predictions.json')
run('package_predictions.py','--task','task3','--input',OUT/'predictions.json',
    '--output',WORK/'submission_task3_qwen3_4b.zip')
print('Tải /kaggle/working/submission_task3_qwen3_4b.zip. Giữ logs/cache riêng tư.')


In [ ]:
# Báo cáo chất lượng vận hành; fallback không đồng nghĩa chẩn đoán tốt.
from collections import Counter
fallbacks = [r for r in pred if r['output'].get('fallback')]
print('Tổng bài:', len(pred), '| Fallback:', len(fallbacks))
print('Fallback theo level:', dict(Counter(r['output']['feedback_level'] for r in fallbacks)))
(OUT/'fallback_report.json').write_text(json.dumps({
    'count':len(fallbacks), 'sample_ids':[r['sample_id'] for r in fallbacks]
},ensure_ascii=False,indent=2),encoding='utf-8')
print('Xem nguyên nhân trong shard-1/responses.jsonl và shard-2/responses.jsonl')
